#Model Train

In [ ]:
!pip install torch torchvision
!pip install scikit-learn
!pip install opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
import pickle
import json
from google.colab import drive
from datetime import datetime
import argparse
from skimage import color
import math

In [ ]:
def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
    print(f"Random seed has been set to {seed}.")

def generate_model_filename(base_dir, params_dict):
    params_str = '_'.join([f"{k}{v}" for k, v in sorted(params_dict.items())])
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    model_dir_name = f"color_model_{params_str}_{timestamp}"
    full_model_dir = os.path.join(base_dir, model_dir_name)
    os.makedirs(full_model_dir, exist_ok=True)

    return full_model_dir

# Dataset class
class ColorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Feature extraction
def extract_features(df):
    X = df[['observed_R', 'observed_G', 'observed_B',
            'red_R', 'red_G', 'red_B',
            'green_R', 'green_G', 'green_B',
            'blue_R', 'blue_G', 'blue_B']].values
    y = df[['true_R', 'true_G', 'true_B']].values
    return X, y

# Dynamic layer creation
def create_encoder_layers(input_dim, hidden_dims, dropout_rate):
    layers = []
    in_features = input_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(in_features, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        in_features = hidden_dim
    return nn.Sequential(*layers)

def create_decoder_layers(latent_dim, hidden_dims, output_dim, dropout_rate):
    layers = []
    in_features = latent_dim
    for hidden_dim in hidden_dims:
        layers.append(nn.Linear(in_features, hidden_dim))
        layers.append(nn.BatchNorm1d(hidden_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))
        in_features = hidden_dim
    # Add output layer
    layers.append(nn.Linear(in_features, output_dim))
    layers.append(nn.Sigmoid())  # Output in 0~1 range
    return nn.Sequential(*layers)

In [ ]:
# AutoEncoder model class
class ColorAutoEncoder(nn.Module):
   def __init__(self, input_dim, encoder_hidden_dims, latent_dim, decoder_hidden_dims, output_dim, dropout_rate):
       super(ColorAutoEncoder, self).__init__()
       # Create encoder
       self.encoder_layers = create_encoder_layers(input_dim, encoder_hidden_dims, dropout_rate)
       self.latent_layer = nn.Linear(encoder_hidden_dims[-1], latent_dim)
       self.latent_activation = nn.ReLU()
       # Create decoder
       self.decoder = create_decoder_layers(latent_dim, decoder_hidden_dims, output_dim, dropout_rate)

   def encode(self, x):
       x = self.encoder_layers(x)
       latent = self.latent_activation(self.latent_layer(x))
       return latent

   def forward(self, x):
       latent = self.encode(x)
       output = self.decoder(latent)
       return output

# Weighted MSE loss function (reflecting human visual sensitivity)
class WeightedRGBLoss(nn.Module):
   def __init__(self):
       super(WeightedRGBLoss, self).__init__()
       # Human vision is most sensitive to green and least sensitive to blue
       self.weights = torch.tensor([0.3, 0.59, 0.11]).view(1, 3)

   def forward(self, predicted, target):
       device = predicted.device
       self.weights = self.weights.to(device)
       squared_diff = (predicted - target) ** 2
       weighted_squared_diff = squared_diff * self.weights
       loss = torch.mean(weighted_squared_diff)
       return loss

# Color similarity loss function
class ColorSimilarityLoss(nn.Module):

   def __init__(self):
       super(ColorSimilarityLoss, self).__init__()

   def forward(self, predicted, target):
       pred_diff_rg = predicted[:, 0] - predicted[:, 1]
       pred_diff_rb = predicted[:, 0] - predicted[:, 2]
       pred_diff_gb = predicted[:, 1] - predicted[:, 2]

       target_diff_rg = target[:, 0] - target[:, 1]
       target_diff_rb = target[:, 0] - target[:, 2]
       target_diff_gb = target[:, 1] - target[:, 2]

       loss_rg = torch.mean((pred_diff_rg - target_diff_rg) ** 2)
       loss_rb = torch.mean((pred_diff_rb - target_diff_rb) ** 2)
       loss_gb = torch.mean((pred_diff_gb - target_diff_gb) ** 2)

       loss = (loss_rg + loss_rb + loss_gb) / 3.0
       return loss

# Composite loss function
class CompositeLoss(nn.Module):

   def __init__(self, alpha=0.6, beta=0.3, gamma=0.1):
       super(CompositeLoss, self).__init__()
       self.mse = nn.MSELoss()
       self.weighted_rgb = WeightedRGBLoss()
       self.color_similarity = ColorSimilarityLoss()
       self.alpha = alpha  # MSE weight
       self.beta = beta    # Weighted RGB loss weight
       self.gamma = gamma  # Color similarity loss weight

   def forward(self, predicted, target):
       mse_loss = self.mse(predicted, target)
       weighted_loss = self.weighted_rgb(predicted, target)
       similarity_loss = self.color_similarity(predicted, target)
       total_loss = (self.alpha * mse_loss +
                     self.beta * weighted_loss +
                     self.gamma * similarity_loss)
       return total_loss, mse_loss, weighted_loss, similarity_loss

# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
   model.train()
   running_total_loss = 0.0
   running_mse_loss = 0.0
   running_weighted_loss = 0.0
   running_similarity_loss = 0.0
   running_mae = 0.0

   for X_batch, y_batch in train_loader:
       X_batch, y_batch = X_batch.to(device), y_batch.to(device)
       optimizer.zero_grad()
       outputs = model(X_batch)

       total_loss, mse_loss, weighted_loss, similarity_loss = criterion(outputs, y_batch)
       mae = torch.mean(torch.abs(outputs - y_batch))

       total_loss.backward()
       optimizer.step()

       running_total_loss += total_loss.item() * X_batch.size(0)
       running_mse_loss += mse_loss.item() * X_batch.size(0)
       running_weighted_loss += weighted_loss.item() * X_batch.size(0)
       running_similarity_loss += similarity_loss.item() * X_batch.size(0)
       running_mae += mae.item() * X_batch.size(0)

   epoch_total_loss = running_total_loss / len(train_loader.dataset)
   epoch_mse_loss = running_mse_loss / len(train_loader.dataset)
   epoch_weighted_loss = running_weighted_loss / len(train_loader.dataset)
   epoch_similarity_loss = running_similarity_loss / len(train_loader.dataset)
   epoch_mae = running_mae / len(train_loader.dataset)
   return epoch_total_loss, epoch_mse_loss, epoch_weighted_loss, epoch_similarity_loss, epoch_mae

# Validation function
def validate(model, val_loader, criterion, device):
   model.eval()
   running_total_loss = 0.0
   running_mse_loss = 0.0
   running_weighted_loss = 0.0
   running_similarity_loss = 0.0
   running_mae = 0.0
   with torch.no_grad():
       for X_batch, y_batch in val_loader:
           X_batch, y_batch = X_batch.to(device), y_batch.to(device)
           outputs = model(X_batch)

           total_loss, mse_loss, weighted_loss, similarity_loss = criterion(outputs, y_batch)
           mae = torch.mean(torch.abs(outputs - y_batch))

           running_total_loss += total_loss.item() * X_batch.size(0)
           running_mse_loss += mse_loss.item() * X_batch.size(0)
           running_weighted_loss += weighted_loss.item() * X_batch.size(0)
           running_similarity_loss += similarity_loss.item() * X_batch.size(0)
           running_mae += mae.item() * X_batch.size(0)

       epoch_total_loss = running_total_loss / len(val_loader.dataset)
       epoch_mse_loss = running_mse_loss / len(val_loader.dataset)
       epoch_weighted_loss = running_weighted_loss / len(val_loader.dataset)
       epoch_similarity_loss = running_similarity_loss / len(val_loader.dataset)
       epoch_mae = running_mae / len(val_loader.dataset)
   return epoch_total_loss, epoch_mse_loss, epoch_weighted_loss, epoch_similarity_loss, epoch_mae

def delta_e(y_true, y_pred):
   """
   Compute the Delta E (color difference) between two RGB colors.
   Converts RGB to Lab space and computes the color difference.
   """
   from skimage.color import rgb2lab
   # Normalize RGB values to [0,1] before conversion
   y_true = np.array(y_true) / 255.0
   y_pred = np.array(y_pred) / 255.0
   lab_true = rgb2lab(y_true.reshape(1, 1, 3)).reshape(3)
   lab_pred = rgb2lab(y_pred.reshape(1, 1, 3)).reshape(3)
   return np.linalg.norm(lab_true - lab_pred)

# Function to calculate additional evaluation metrics
def calculate_metrics(y_true, y_pred):
   """
   Calculate additional evaluation metrics:
   - R² Score
   - RMSE
   - MAPE
   - Color similarity (Delta E)
   """
   # R² Score
   r2 = r2_score(y_true, y_pred)
   # RMSE
   rmse = np.sqrt(mean_squared_error(y_true, y_pred))
   # MAPE (avoid division by zero)
   mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
   # ΔE for each color and take mean & median
   delta_e_values = [delta_e(y_true[i], y_pred[i]) for i in range(len(y_true))]
   mean_delta_e = np.mean(delta_e_values)
   median_delta_e = np.median(delta_e_values)
   return {
       "R2_Score": r2,
       "RMSE": rmse,
       "MAPE": mape,
       "Mean_Delta_E": mean_delta_e,
       "Median_Delta_E": median_delta_e
   }

In [ ]:
# Main function
def main(args):
   set_random_seed(args.seed)
   device = torch.device("cuda" if torch.cuda.is_available() and not args.no_cuda else "cpu")
   print(f"Device in use: {device}")

   # Set reference colors (standards)
   RED_REFERENCE = (230, 50, 35)
   GREEN_REFERENCE = (120, 250, 80)
   BLUE_REFERENCE = (0, 0, 250)
   print("=== Color Calibration AutoEncoder Model (PyTorch) ===")
   print(f"Reference RED RGB: {RED_REFERENCE}")
   print(f"Reference GREEN RGB: {GREEN_REFERENCE}")
   print(f"Reference BLUE RGB: {BLUE_REFERENCE}")

   # 1. Mount Google Drive
   if args.use_drive:
       print("\n1. Mounting Google Drive")
       drive.mount('/content/drive')

   # 2. Load data
   print("\n2. Loading data...")
   try:
       train_df = pd.read_csv(args.train_path)
       val_df = pd.read_csv(args.val_path)
       test_df = pd.read_csv(args.test_path)
       print(f"Training data size: {train_df.shape}")
       print(f"Validation data size: {val_df.shape}")
       print(f"Test data size: {test_df.shape}")
   except Exception as e:
       print(f"Error loading data: {e}")
       raise

   # 3. Data preprocessing
   print("\n3. Data preprocessing")
   # Handle missing values
   for df_name, df in [("Training", train_df), ("Validation", val_df), ("Test", test_df)]:
       missing_values = df.isnull().sum().sum()
       if missing_values > 0:
           print(f"Processing missing values in {df_name} data...")
           df.dropna(inplace=True)
           print(f"{df_name} data size after removing missing values: {df.shape}")
   # Extract features
   X_train, y_train = extract_features(train_df)
   X_val, y_val = extract_features(val_df)
   X_test, y_test = extract_features(test_df)
   # Scaling
   X_scaler = MinMaxScaler()
   y_scaler = MinMaxScaler()
   X_scaler.fit(X_train)
   y_scaler.fit(y_train)
   X_train_scaled = X_scaler.transform(X_train)
   X_val_scaled = X_scaler.transform(X_val)
   X_test_scaled = X_scaler.transform(X_test)
   y_train_scaled = y_scaler.transform(y_train)
   y_val_scaled = y_scaler.transform(y_val)
   y_test_scaled = y_scaler.transform(y_test)
   print(f"Scaling completed")
   print(f"Training input data shape: {X_train_scaled.shape}")
   print(f"Training target data shape: {y_train_scaled.shape}")

   # 4. Create datasets and dataloaders
   train_dataset = ColorDataset(X_train_scaled, y_train_scaled)
   val_dataset = ColorDataset(X_val_scaled, y_val_scaled)
   test_dataset = ColorDataset(X_test_scaled, y_test_scaled)
   train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
   val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
   test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

   # 5. Model configuration
   # Hidden layer configuration
   encoder_hidden_dims = [128, 64] if args.encoder_layers == 2 else [128, 64, 32]
   decoder_hidden_dims = [64, 128] if args.decoder_layers == 2 else [32, 64, 128]
   # Create model parameters dictionary
   model_params = {
       'e': args.epochs,
       'b': args.batch_size,
       'lr': args.learning_rate,
       'lat': args.latent_dim,
       'enc': args.encoder_layers,
       'dec': args.decoder_layers,
       'drop': int(args.dropout_rate * 100),  # Multiply by 100 to avoid decimal points
       'loss': 'composite',
       'alpha': args.loss_alpha,
       'beta': args.loss_beta,
       'gamma': args.loss_gamma
   }
   # Initialize model
   input_dim = X_train_scaled.shape[1]  # 12 (4 colors * RGB)
   output_dim = y_train_scaled.shape[1]  # 3 (RGB)
   model = ColorAutoEncoder(
       input_dim,
       encoder_hidden_dims,
       args.latent_dim,
       decoder_hidden_dims,
       output_dim,
       args.dropout_rate
   ).to(device)
   print("\n5. AutoEncoder Model Implementation")
   print(model)

   # 6. Training setup
   criterion = CompositeLoss(alpha=args.loss_alpha, beta=args.loss_beta, gamma=args.loss_gamma)
   optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
   scheduler = optim.lr_scheduler.ReduceLROnPlateau(
       optimizer, mode='min', factor=0.5, patience=5, min_lr=0.00001
   )
   print(f"\n6. Composite Loss Function Setup")
   print(f"Alpha (MSE): {args.loss_alpha}")
   print(f"Beta (Weighted RGB): {args.loss_beta}")
   print(f"Gamma (Color Similarity): {args.loss_gamma}")

   # 7. Set model save path
   os.makedirs(args.output_dir, exist_ok=True)
   model_dir = generate_model_filename(args.output_dir, model_params)
   os.makedirs(model_dir, exist_ok=True)

   model_filename = f"model_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.pt"
   history_filename = f"history_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.png"
   metrics_filename = f"metrics_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.json"
   scalers_filename = f"scalers_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.pkl"

   model_path = os.path.join(model_dir, model_filename)
   history_path = os.path.join(model_dir, history_filename)
   metrics_path = os.path.join(model_dir, metrics_filename)
   scaler_path = os.path.join(model_dir, scalers_filename)

   # 8. Start model training
   print("\n7. Starting model training")
   # Training history
   train_total_losses = []
   train_mse_losses = []
   train_weighted_losses = []
   train_similarity_losses = []
   train_maes = []
   val_total_losses = []
   val_mse_losses = []
   val_weighted_losses = []
   val_similarity_losses = []
   val_maes = []
   learning_rates = []
   # Early stopping setup
   best_val_loss = float('inf')
   patience_counter = 0
   print(f"Starting training for {args.epochs} epochs...")
   for epoch in range(args.epochs):
       # Train
       train_total_loss, train_mse_loss, train_weighted_loss, train_similarity_loss, train_mae = train_epoch(
           model, train_loader, criterion, optimizer, device
       )
       # Validate
       val_total_loss, val_mse_loss, val_weighted_loss, val_similarity_loss, val_mae = validate(
           model, val_loader, criterion, device
       )
       # Save current learning rate
       current_lr = optimizer.param_groups[0]['lr']
       learning_rates.append(current_lr)
       # Update learning rate scheduler
       scheduler.step(val_total_loss)
       # Record results
       train_total_losses.append(train_total_loss)
       train_mse_losses.append(train_mse_loss)
       train_weighted_losses.append(train_weighted_loss)
       train_similarity_losses.append(train_similarity_loss)
       train_maes.append(train_mae)
       val_total_losses.append(val_total_loss)
       val_mse_losses.append(val_mse_loss)
       val_weighted_losses.append(val_weighted_loss)
       val_similarity_losses.append(val_similarity_loss)
       val_maes.append(val_mae)
       # Save model (only if validation loss improves)
       if val_total_loss < best_val_loss:
           best_val_loss = val_total_loss
           # Save model with metadata
           torch.save({
               'epoch': epoch,
               'model_state_dict': model.state_dict(),
               'optimizer_state_dict': optimizer.state_dict(),
               'val_loss': val_total_loss,
               'train_loss': train_total_loss,
               'model_params': model_params
           }, model_path)
           patience_counter = 0
           print(f"Epoch {epoch+1}/{args.epochs}, Improved model saved: {os.path.basename(model_path)}")
       else:
           patience_counter += 1
       # Print epoch results
       print(f"Epoch {epoch+1}/{args.epochs}, "
             f"Train Loss: {train_total_loss:.4f}, Train MAE: {train_mae:.4f}, "
             f"Val Loss: {val_total_loss:.4f}, Val MAE: {val_mae:.4f}, "
             f"LR: {current_lr:.6f}")
       # Check for early stopping
       if patience_counter >= args.patience:
           print(f"Early stopping: Validation loss has not improved for {args.patience} consecutive epochs.")
           break
   print("Training completed!")

   # 9. Load best model
   checkpoint = torch.load(model_path)
   model.load_state_dict(checkpoint['model_state_dict'])
   model.eval()
   print(f"\nLoaded best model (Epoch {checkpoint['epoch']+1}, Validation Loss: {checkpoint['val_loss']:.6f})")

   # 10. Evaluate model on test data
   test_total_loss, test_mse_loss, test_weighted_loss, test_similarity_loss, test_mae = validate(
       model, test_loader, criterion, device
   )
   print(f"\nTest total loss: {test_total_loss:.4f}")
   print(f"Test MSE loss: {test_mse_loss:.4f}")
   print(f"Test weighted RGB loss: {test_weighted_loss:.4f}")
   print(f"Test color similarity loss: {test_similarity_loss:.4f}")
   print(f"Test MAE: {test_mae:.4f}")

   # 11. Visualize training results
   plt.figure(figsize=(15, 10))
   # Total loss graph
   plt.subplot(2, 2, 1)
   plt.plot(train_total_losses, label='train_loss')
   plt.plot(val_total_losses, label='val_loss')
   plt.title('Total Loss Function')
   plt.xlabel('Epoch')
   plt.ylabel('Loss')
   plt.legend()
   # MSE loss graph
   plt.subplot(2, 2, 2)
   plt.plot(train_mse_losses, label='train_mse')
   plt.plot(val_mse_losses, label='val_mse')
   plt.title('MSE loss')
   plt.xlabel('Epoch')
   plt.ylabel('Loss')
   plt.legend()
   # Weighted RGB & Color similarity loss graph
   plt.subplot(2, 2, 3)
   plt.plot(train_weighted_losses, label='train_weighted')
   plt.plot(val_weighted_losses, label='val_weighted')
   plt.plot(train_similarity_losses, label='train_similarity')
   plt.plot(val_similarity_losses, label='val_similarity')
   plt.title('"Weighted RGB & Color Similarity Loss"')
   plt.xlabel('Epoch')
   plt.ylabel('Loss')
   plt.legend()
   # MAE & Learning rate graph
   plt.subplot(2, 2, 4)
   ax1 = plt.gca()
   ax1.plot(train_maes, 'b-', label='train_mae')
   ax1.plot(val_maes, 'g-', label='val_mae')
   ax1.set_xlabel('Epoch')
   ax1.set_ylabel('MAE', color='b')
   ax1.tick_params('y', colors='b')
   ax2 = ax1.twinx()
   ax2.plot(learning_rates, 'r-', label='learning_rate')
   ax2.set_ylabel('Learning Rate', color='r')
   ax2.tick_params('y', colors='r')
   # Combine legends from both axes
   lines1, labels1 = ax1.get_legend_handles_labels()
   lines2, labels2 = ax2.get_legend_handles_labels()
   ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
   plt.title('MAE & LR')
   plt.tight_layout()
   # Save training graph to model directory
   plt.savefig(history_path)
   print(f"Training result graph saved: {history_path}")

   # 12. Calculate additional test evaluation metrics and visualize colors
   print("\n12. Calculating detailed evaluation metrics and visualizing colors for test data...")
   with torch.no_grad():
       model.eval()
       all_true_rgb = []
       all_pred_rgb = []
       for X_batch, y_batch in test_loader:
           X_batch = X_batch.to(device)
           y_batch = y_batch.cpu().numpy()  # Original scaled target values
           # Run prediction
           outputs = model(X_batch).cpu().numpy()
           # Inverse scaling
           y_batch_original = y_scaler.inverse_transform(y_batch)
           outputs_original = y_scaler.inverse_transform(outputs)
           # Store results
           all_true_rgb.append(y_batch_original)
           all_pred_rgb.append(outputs_original)
       all_true_rgb = np.vstack(all_true_rgb)
       all_pred_rgb = np.vstack(all_pred_rgb)

       metrics = calculate_metrics(all_true_rgb, all_pred_rgb)

       print("\nEvaluation Metric Results:")
       print(f"R²_Score : {metrics['R2_Score']:.4f}")
       print(f"RMSE : {metrics['RMSE']:.4f}")
       print(f"MAPE: {metrics['MAPE']:.4f}")
       print(f"Mean_Delta_E: {metrics['Mean_Delta_E']:.4f}")
       print(f"Median_Delta_E: {metrics['Median_Delta_E']:.4f}")

       # Save metrics to model directory
       with open(metrics_path, 'w') as f:
           json.dump(metrics, f, indent=2, default=str)
       print(f"Evaluation metrics saved: {metrics_path}")
       # Color visualization - comparing actual colors and predicted colors for test samples
       print("\nVisualizing colors for test samples...")

       colors_vis_filename = f"color_vis_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.png"
       colors_vis_path = os.path.join(model_dir, colors_vis_filename)
       # Sort by color difference
       color_diffs = np.sqrt(np.sum((all_true_rgb - all_pred_rgb) ** 2, axis=1))
       sorted_indices = np.argsort(color_diffs)  # Sort in ascending order
       # Number of samples to visualize
       n_samples = min(50, len(all_true_rgb))
       sample_indices = sorted_indices[:n_samples]
       # Visualization setup
       fig, axes = plt.subplots(n_samples, 2, figsize=(8, n_samples * 0.8))
       fig.suptitle('Comparison of Color Restoration Results (Sorted by Smallest Color Difference)', fontsize=10)

       if n_samples > 1:
           axes[0, 0].set_title('Actual Color\n(Original RGB)')
           axes[0, 1].set_title('Predicted Color\n(Original RGB)')
       else:
           axes[0].set_title('Actual Color\n(Original RGB)')
           axes[1].set_title('Predicted Color\n(Original RGB)')

       # Color comparison visualization
       for i, idx in enumerate(sample_indices):
           # Actual target RGB values
           true_rgb = np.clip(all_true_rgb[idx], 0, 255).astype(np.uint8)
           # Predicted RGB values
           pred_rgb = np.clip(all_pred_rgb[idx], 0, 255).astype(np.uint8)
           # Normalize RGB values to [0, 1] range
           true_rgb_norm = true_rgb / 255.0
           pred_rgb_norm = pred_rgb / 255.0

           if n_samples > 1:
               axes[i, 0].add_patch(plt.Rectangle((0, 0), 1, 1, color=true_rgb_norm))
               axes[i, 1].add_patch(plt.Rectangle((0, 0), 1, 1, color=pred_rgb_norm))
               for j in range(2):
                   axes[i, j].set_xticks([])
                   axes[i, j].set_yticks([])
                   axes[i, j].set_xlim(0, 1)
                   axes[i, j].set_ylim(0, 1)
               axes[i, 0].text(0.5, -0.1, f"({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})",
                              ha='center', va='center', fontsize=8, transform=axes[i, 0].transAxes)
               axes[i, 1].text(0.5, -0.1, f"({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})",
                              ha='center', va='center', fontsize=8, transform=axes[i, 1].transAxes)
           else:
               # For single sample
               axes[0].add_patch(plt.Rectangle((0, 0), 1, 1, color=true_rgb_norm))
               axes[1].add_patch(plt.Rectangle((0, 0), 1, 1, color=pred_rgb_norm))
               for j in range(2):
                   axes[j].set_xticks([])
                   axes[j].set_yticks([])
                   axes[j].set_xlim(0, 1)
                   axes[j].set_ylim(0, 1)
               axes[0].text(0.5, -0.1, f"({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})",
                           ha='center', va='center', fontsize=8, transform=axes[0].transAxes)
               axes[1].text(0.5, -0.1, f"({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})",
                           ha='center', va='center', fontsize=8, transform=axes[1].transAxes)
       plt.tight_layout()
       plt.subplots_adjust(top=0.95)
       plt.savefig(colors_vis_path, dpi=300, bbox_inches='tight')
       plt.close(fig)
       print(f"Color visualization saved: {colors_vis_path}")

       # Save color sample data to CSV
       samples_data = []
       for idx in sample_indices:
           true_rgb = all_true_rgb[idx]
           pred_rgb = all_pred_rgb[idx]
           color_diff = color_diffs[idx]
           samples_data.append({
               'sample_idx': idx,
               'true_r': int(np.clip(true_rgb[0], 0, 255)),
               'true_g': int(np.clip(true_rgb[1], 0, 255)),
               'true_b': int(np.clip(true_rgb[2], 0, 255)),
               'pred_r': int(np.clip(pred_rgb[0], 0, 255)),
               'pred_g': int(np.clip(pred_rgb[1], 0, 255)),
               'pred_b': int(np.clip(pred_rgb[2], 0, 255)),
               'color_diff': float(color_diff)
           })

       # Save to CSV file
       samples_df = pd.DataFrame(samples_data)
       samples_csv_filename = f"color_samples_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.csv"
       samples_csv_path = os.path.join(model_dir, samples_csv_filename)
       samples_df.to_csv(samples_csv_path, index=False)
       print(f"Sample color data saved as CSV: {samples_csv_path}")

       # 13. Save scalers
       with open(scaler_path, 'wb') as f:
           pickle.dump({'X_scaler': X_scaler, 'y_scaler': y_scaler}, f)
       print(f"Scalers saved: {scaler_path}")

       # 14. Results summary
       print("\n====== Model Training and Evaluation Complete ======")
       print(f"Model file: {os.path.basename(model_path)}")
       print(f"Total epochs: {checkpoint['epoch']+1}/{args.epochs}")
       print(f"Final validation loss: {checkpoint['val_loss']:.6f}")
       print(f"Test MAE: {test_mae:.6f}")
       print(f"R²_Score : {metrics['R2_Score']:.4f}")
       print(f"RMSE : {metrics['RMSE']:.4f}")
       print(f"MAPE: {metrics['MAPE']:.4f}")
       print(f"Mean_Delta_E: {metrics['Mean_Delta_E']:.4f}")
       print(f"Median_Delta_E: {metrics['Median_Delta_E']:.4f}")
       print("\nVisualization files:")
       print(f"1. Training results graph: {os.path.basename(history_path)}")
       print(f"2. Color restoration comparison: {os.path.basename(colors_vis_path)}")
       print(f"3. Sample color data: {os.path.basename(samples_csv_path)}")
       print("=====================================")

In [ ]:
if __name__ == "__main__":
   parser = argparse.ArgumentParser(description='Color Calibration AutoEncoder Model Training Script')
   # Data related arguments
   parser.add_argument('--train_path', type=str,
                    default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_train_split.csv',
                    help='Training data path')
   parser.add_argument('--val_path', type=str,
                    default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_val_split.csv',
                    help='Validation data path')
   parser.add_argument('--test_path', type=str,
                    default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_test_split.csv',
                    help='Test data path')
   parser.add_argument('--output_dir', type=str,
                    default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/output/model/added_data',
                    help='Model save directory')
   parser.add_argument('--use_drive', action='store_true', help='Use Google Drive')
   # Model configuration arguments
   parser.add_argument('--latent_dim', type=int, default=32, help='Latent space dimension')
   parser.add_argument('--encoder_layers', type=int, default=3, choices=[2, 3], help='Number of encoder hidden layers')
   parser.add_argument('--decoder_layers', type=int, default=3, choices=[2, 3], help='Number of decoder hidden layers')
   parser.add_argument('--dropout_rate', type=float, default=0.2, help='Dropout rate')
   # Training arguments
   parser.add_argument('--epochs', type=int, default=100, help='Number of training epochs')
   parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
   parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate')
   parser.add_argument('--patience', type=int, default=30, help='Early stopping patience')
   parser.add_argument('--seed', type=int, default=42, help='Random seed')
   parser.add_argument('--no_cuda', action='store_true', help='Do not use CUDA')
   # Loss function weights
   parser.add_argument('--loss_alpha', type=float, default=0.2, help='MSE loss weight')
   parser.add_argument('--loss_beta', type=float, default=0.5, help='Weighted RGB loss weight')
   parser.add_argument('--loss_gamma', type=float, default=0.3, help='Color similarity loss weight')
   # Modifications for running in Colab
   import sys
   if 'google.colab' in sys.modules:
       args = parser.parse_args([
           '--train_path', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_train_split.csv',
           '--val_path', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_val_split.csv',
           '--test_path', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/added_test_split.csv',
           '--output_dir', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/output/model/added_data',
           '--use_drive',
           '--epochs', '40',
           '--batch_size', '32',
           '--learning_rate', '0.005',
           '--latent_dim', '256',
           '--encoder_layers', '3',
           '--decoder_layers', '2',
           '--loss_alpha', '0.2',
           '--loss_beta', '0.5',
           '--loss_gamma', '0.3',
           '--dropout_rate', '0.1'
       ])
   else:
       args = parser.parse_args()
   # Verify that alpha + beta + gamma = 1
   sum_weights = args.loss_alpha + args.loss_beta + args.loss_gamma
   if abs(sum_weights - 1.0) > 1e-5:
       print(f"Warning: Sum of loss function weights is not 1 (current: {sum_weights})")
       # Normalize weights
       args.loss_alpha /= sum_weights
       args.loss_beta /= sum_weights
       args.loss_gamma /= sum_weights
       print(f"Weights have been normalized: alpha={args.loss_alpha:.4f}, beta={args.loss_beta:.4f}, gamma={args.loss_gamma:.4f}")
   main(args)

# K-fold

In [ ]:
!pip install torch torchvision
!pip install scikit-learn
!pip install opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import math
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold
from datetime import datetime
import matplotlib.pyplot as plt
from skimage import color
from sklearn.model_selection import train_test_split
import sys
import time
import argparse

In [ ]:
# Dataset class
class ColorDataset(Dataset):
   def __init__(self, X, y):
       self.X = torch.tensor(X, dtype=torch.float32)
       self.y = torch.tensor(y, dtype=torch.float32)
   def __len__(self):
       return len(self.X)
   def __getitem__(self, idx):
       return self.X[idx], self.y[idx]

# Feature extraction
def extract_features(df):
   """Extract input features and target values from DataFrame."""
   X = df[['observed_R', 'observed_G', 'observed_B',
           'red_R', 'red_G', 'red_B',
           'green_R', 'green_G', 'green_B',
           'blue_R', 'blue_G', 'blue_B']].values
   y = df[['true_R', 'true_G', 'true_B']].values
   return X, y

def set_random_seed(seed):
   """Set random seed for reproducibility."""
   np.random.seed(seed)
   torch.manual_seed(seed)
   if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.backends.cudnn.deterministic = True
   print(f"Random seed set to {seed}.")

def generate_model_filename(base_dir, params_dict):
   """Generate model directory name."""
   params_str = '_'.join([f"{k}{v}" for k, v in sorted(params_dict.items())])
   timestamp = datetime.now().strftime("%Y%m%d_%H%M")
   model_dir_name = f"color_model_{params_str}_{timestamp}"
   full_model_dir = os.path.join(base_dir, model_dir_name)
   os.makedirs(full_model_dir, exist_ok=True)
   return full_model_dir

# Dynamic layer creation
def create_encoder_layers(input_dim, hidden_dims, dropout_rate):
   """Create encoder layers with batch normalization and dropout."""
   layers = []
   in_features = input_dim
   for hidden_dim in hidden_dims:
       layers.append(nn.Linear(in_features, hidden_dim))
       layers.append(nn.BatchNorm1d(hidden_dim))
       layers.append(nn.ReLU())
       layers.append(nn.Dropout(dropout_rate))
       in_features = hidden_dim
   return nn.Sequential(*layers)

def create_decoder_layers(latent_dim, hidden_dims, output_dim, dropout_rate):
   """Create decoder layers with batch normalization and dropout."""
   layers = []
   in_features = latent_dim
   for hidden_dim in hidden_dims:
       layers.append(nn.Linear(in_features, hidden_dim))
       layers.append(nn.BatchNorm1d(hidden_dim))
       layers.append(nn.ReLU())
       layers.append(nn.Dropout(dropout_rate))
       in_features = hidden_dim
   layers.append(nn.Linear(in_features, output_dim))
   layers.append(nn.Sigmoid())  # Output in 0~1 range
   return nn.Sequential(*layers)

class ColorAutoEncoder(nn.Module):
   def __init__(self, input_dim, encoder_hidden_dims, latent_dim, decoder_hidden_dims, output_dim, dropout_rate):
       super(ColorAutoEncoder, self).__init__()
       # Create encoder
       self.encoder_layers = create_encoder_layers(input_dim, encoder_hidden_dims, dropout_rate)
       self.latent_layer = nn.Linear(encoder_hidden_dims[-1], latent_dim)
       self.latent_activation = nn.ReLU()
       # Create decoder
       self.decoder = create_decoder_layers(latent_dim, decoder_hidden_dims, output_dim, dropout_rate)

   def encode(self, x):
       x = self.encoder_layers(x)
       latent = self.latent_activation(self.latent_layer(x))
       return latent

   def forward(self, x):
       latent = self.encode(x)
       output = self.decoder(latent)
       return output

# Loss Functions
class WeightedRGBLoss(nn.Module):
   def __init__(self):
       super(WeightedRGBLoss, self).__init__()
       # Human vision is most sensitive to green and least sensitive to blue
       self.weights = torch.tensor([0.3, 0.59, 0.11]).view(1, 3)
   def forward(self, predicted, target):
       device = predicted.device
       self.weights = self.weights.to(device)
       squared_diff = (predicted - target) ** 2
       weighted_squared_diff = squared_diff * self.weights
       loss = torch.mean(weighted_squared_diff)
       return loss

class ColorSimilarityLoss(nn.Module):
   def __init__(self):
       super(ColorSimilarityLoss, self).__init__()

   def forward(self, predicted, target):
       pred_diff_rg = predicted[:, 0] - predicted[:, 1]
       pred_diff_rb = predicted[:, 0] - predicted[:, 2]
       pred_diff_gb = predicted[:, 1] - predicted[:, 2]
       target_diff_rg = target[:, 0] - target[:, 1]
       target_diff_rb = target[:, 0] - target[:, 2]
       target_diff_gb = target[:, 1] - target[:, 2]

       loss_rg = torch.mean((pred_diff_rg - target_diff_rg) ** 2)
       loss_rb = torch.mean((pred_diff_rb - target_diff_rb) ** 2)
       loss_gb = torch.mean((pred_diff_gb - target_diff_gb) ** 2)

       loss = (loss_rg + loss_rb + loss_gb) / 3.0
       return loss

class CompositeLoss(nn.Module):
   def __init__(self, alpha=0.6, beta=0.3, gamma=0.1):
       super(CompositeLoss, self).__init__()
       self.mse = nn.MSELoss()
       self.weighted_rgb = WeightedRGBLoss()
       self.color_similarity = ColorSimilarityLoss()
       self.alpha = alpha  # MSE weight
       self.beta = beta    # Weighted RGB loss weight
       self.gamma = gamma  # Color similarity loss weight
   def forward(self, predicted, target):
       mse_loss = self.mse(predicted, target)
       weighted_loss = self.weighted_rgb(predicted, target)
       similarity_loss = self.color_similarity(predicted, target)

       total_loss = (self.alpha * mse_loss +
                     self.beta * weighted_loss +
                     self.gamma * similarity_loss)
       return total_loss, mse_loss, weighted_loss, similarity_loss

# Training and Evaluation Function
def train_epoch(model, train_loader, criterion, optimizer, device):
   """Train the model for one epoch."""
   model.train()
   running_total_loss = 0.0
   running_mse_loss = 0.0
   running_weighted_loss = 0.0
   running_similarity_loss = 0.0
   running_mae = 0.0
   for X_batch, y_batch in train_loader:
       X_batch, y_batch = X_batch.to(device), y_batch.to(device)
       optimizer.zero_grad()
       outputs = model(X_batch)

       total_loss, mse_loss, weighted_loss, similarity_loss = criterion(outputs, y_batch)

       mae = torch.mean(torch.abs(outputs - y_batch))  # Calculate MAE (for monitoring)

       total_loss.backward()
       optimizer.step()

       running_total_loss += total_loss.item() * X_batch.size(0)
       running_mse_loss += mse_loss.item() * X_batch.size(0)
       running_weighted_loss += weighted_loss.item() * X_batch.size(0)
       running_similarity_loss += similarity_loss.item() * X_batch.size(0)
       running_mae += mae.item() * X_batch.size(0)

   dataset_size = len(train_loader.dataset)
   epoch_total_loss = running_total_loss / dataset_size
   epoch_mse_loss = running_mse_loss / dataset_size
   epoch_weighted_loss = running_weighted_loss / dataset_size
   epoch_similarity_loss = running_similarity_loss / dataset_size
   epoch_mae = running_mae / dataset_size
   return epoch_total_loss, epoch_mse_loss, epoch_weighted_loss, epoch_similarity_loss, epoch_mae

def validate(model, val_loader, criterion, device):
   """Evaluate the model on validation or test data."""
   model.eval()
   running_total_loss = 0.0
   running_mse_loss = 0.0
   running_weighted_loss = 0.0
   running_similarity_loss = 0.0
   running_mae = 0.0
   with torch.no_grad():
       for X_batch, y_batch in val_loader:
           X_batch, y_batch = X_batch.to(device), y_batch.to(device)
           outputs = model(X_batch)

           total_loss, mse_loss, weighted_loss, similarity_loss = criterion(outputs, y_batch)

           mae = torch.mean(torch.abs(outputs - y_batch)) # Calculate MAE (for monitoring)

           running_total_loss += total_loss.item() * X_batch.size(0)
           running_mse_loss += mse_loss.item() * X_batch.size(0)
           running_weighted_loss += weighted_loss.item() * X_batch.size(0)
           running_similarity_loss += similarity_loss.item() * X_batch.size(0)
           running_mae += mae.item() * X_batch.size(0)

       dataset_size = len(val_loader.dataset)
       epoch_total_loss = running_total_loss / dataset_size
       epoch_mse_loss = running_mse_loss / dataset_size
       epoch_weighted_loss = running_weighted_loss / dataset_size
       epoch_similarity_loss = running_similarity_loss / dataset_size
       epoch_mae = running_mae / dataset_size
   return epoch_total_loss, epoch_mse_loss, epoch_weighted_loss, epoch_similarity_loss, epoch_mae

def calculate_metrics(y_true, y_pred):
   """
   Calculate additional evaluation metrics:
   - R² values (per channel)
   - RMSE (per channel and overall)
   - Color similarity (Delta E)
   """
   r2_r = float(r2_score(y_true[:, 0], y_pred[:, 0]))
   r2_g = float(r2_score(y_true[:, 1], y_pred[:, 1]))
   r2_b = float(r2_score(y_true[:, 2], y_pred[:, 2]))
   # Calculate RMSE per channel
   rmse_r = float(math.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0])))
   rmse_g = float(math.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1])))
   rmse_b = float(math.sqrt(mean_squared_error(y_true[:, 2], y_pred[:, 2])))
   # Calculate overall RMSE
   rmse_total = float(math.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))))
   # Calculate Delta E (color similarity)
   y_true_rgb = np.clip(y_true / 255.0, 0, 1)
   y_pred_rgb = np.clip(y_pred / 255.0, 0, 1)
   # Convert RGB to LAB color space
   try:
       y_true_lab = color.rgb2lab(y_true_rgb.reshape(-1, 1, 3)).reshape(-1, 3)
       y_pred_lab = color.rgb2lab(y_pred_rgb.reshape(-1, 1, 3)).reshape(-1, 3)
       # Calculate color distance (Delta E)
       delta_e = np.sqrt(np.sum(np.square(y_true_lab - y_pred_lab), axis=1))
       delta_e_mean = float(np.mean(delta_e))
   except:
       # Return -1 if conversion fails
       delta_e_mean = -1.0
   metrics = {
       'r2_r': r2_r,
       'r2_g': r2_g,
       'r2_b': r2_b,
       'rmse_r': rmse_r,
       'rmse_g': rmse_g,
       'rmse_b': rmse_b,
       'rmse_total': rmse_total,
       'delta_e_mean': delta_e_mean
   }
   return metrics

In [ ]:
# Main Function
def main_from_master_dataset(args):
   """
   Train a color calibration model using K-Fold cross-validation starting from a master dataset.
   Args:
       args: Command line arguments
   """

   set_random_seed(args.seed)
   device = torch.device("cuda" if torch.cuda.is_available() and not args.no_cuda else "cpu")
   print(f"Device in use: {device}")

   # Set reference colors (standards)
   RED_REFERENCE = (230, 50, 35)
   GREEN_REFERENCE = (120, 250, 80)
   BLUE_REFERENCE = (0, 0, 250)
   print("=== Color Calibration AutoEncoder Model (PyTorch) - K-Fold Cross Validation from Master Dataset ===")
   print(f"Reference RED RGB: {RED_REFERENCE}")
   print(f"Reference GREEN RGB: {GREEN_REFERENCE}")
   print(f"Reference BLUE RGB: {BLUE_REFERENCE}")

   # 1. Mount Google Drive
   if args.use_drive:
       try:
           from google.colab import drive
           drive.mount('/content/drive')
           print("\nGoogle Drive mounted successfully")
       except ImportError:
           print("Not in Google Colab environment or unable to import drive module.")

   # 2. Load master dataset
   print("\nLoading master dataset...")
   try:
       master_df = pd.read_csv(args.master_path)
       print(f"Master dataset size: {master_df.shape}")
   except Exception as e:
       print(f"Error loading master dataset: {e}")
       raise

   # 3. Data preprocessing
   print("\nPreprocessing data...")
   # Handle missing values
   missing_values = master_df.isnull().sum().sum()
   if missing_values > 0:
       print(f"Processing missing values... (count: {missing_values})")
       master_df.dropna(inplace=True)
       print(f"Dataset size after removing missing values: {master_df.shape}")

   # 4. Dataset splitting: first separate test set, use remainder for K-Fold
   print("\nSplitting dataset...")

   # Separate test set from entire dataset
   train_val_df, test_df = train_test_split(
       master_df, test_size=args.test_size, random_state=args.seed
   )
   print(f"Train/validation data size: {train_val_df.shape} (for K-Fold)")
   print(f"Test data size: {test_df.shape}")

   X_test, y_test = extract_features(test_df)

   # 5. K-Fold setup
   print(f"\nK-Fold cross-validation setup (k={args.n_folds})")
   kf = KFold(n_splits=args.n_folds, shuffle=True, random_state=args.seed)
   X_train_val, y_train_val = extract_features(train_val_df)

   # 6. Create results directory
   output_dir = os.path.join(args.output_dir, f"master_kfold_{args.n_folds}_{datetime.now().strftime('%Y%m%d_%H%M')}")
   os.makedirs(output_dir, exist_ok=True)

   # 7. Execute K-Fold cross-validation
   fold_results = []
   fold_test_losses = []
   fold_test_maes = []
   fold_r2_scores = []
   fold_rmse_scores = []
   fold_delta_e_means = []
   for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
       print(f"\n{'='*50}")
       print(f"Fold {fold+1}/{args.n_folds}")
       print(f"{'='*50}")

       X_train_fold, X_val_fold = X_train_val[train_idx], X_train_val[val_idx]
       y_train_fold, y_val_fold = y_train_val[train_idx], y_train_val[val_idx]
       print(f"Training data size: {len(X_train_fold)}")
       print(f"Validation data size: {len(X_val_fold)}")
       # Scaling
       X_scaler = MinMaxScaler()
       y_scaler = MinMaxScaler()
       X_scaler.fit(X_train_fold)
       y_scaler.fit(y_train_fold)
       X_train_scaled = X_scaler.transform(X_train_fold)
       X_val_scaled = X_scaler.transform(X_val_fold)
       X_test_scaled = X_scaler.transform(X_test)
       y_train_scaled = y_scaler.transform(y_train_fold)
       y_val_scaled = y_scaler.transform(y_val_fold)
       y_test_scaled = y_scaler.transform(y_test)

       train_dataset = ColorDataset(X_train_scaled, y_train_scaled)
       val_dataset = ColorDataset(X_val_scaled, y_val_scaled)
       test_dataset = ColorDataset(X_test_scaled, y_test_scaled)
       train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
       val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
       test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

       # Hidden layer configuration
       encoder_hidden_dims = [128, 64] if args.encoder_layers == 2 else [128, 64, 32]
       decoder_hidden_dims = [64, 128] if args.decoder_layers == 2 else [32, 64, 128]
       # Create model parameters dictionary
       model_params = {
           'e': args.epochs,
           'b': args.batch_size,
           'lr': args.learning_rate,
           'lat': args.latent_dim,
           'enc': args.encoder_layers,
           'dec': args.decoder_layers,
           'drop': int(args.dropout_rate * 100),
           'loss': 'composite',
           'alpha': args.loss_alpha,
           'beta': args.loss_beta,
           'gamma': args.loss_gamma,
           'fold': fold+1
       }
       # Initialize model
       input_dim = X_train_scaled.shape[1]
       output_dim = y_train_scaled.shape[1]
       model = ColorAutoEncoder(
           input_dim,
           encoder_hidden_dims,
           args.latent_dim,
           decoder_hidden_dims,
           output_dim,
           args.dropout_rate
       ).to(device)
       print("\nModel architecture:")
       print(model)
       # Loss function and optimizer setup
       criterion = CompositeLoss(alpha=args.loss_alpha, beta=args.loss_beta, gamma=args.loss_gamma)
       optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)
       scheduler = optim.lr_scheduler.ReduceLROnPlateau(
           optimizer, mode='min', factor=0.5, patience=5, min_lr=0.00001
       )
       print(f"\nComposite loss function setup:")
       print(f"Alpha (MSE): {args.loss_alpha}")
       print(f"Beta (Weighted RGB): {args.loss_beta}")
       print(f"Gamma (Color Similarity): {args.loss_gamma}")

       fold_dir = os.path.join(output_dir, f"fold_{fold+1}")
       os.makedirs(fold_dir, exist_ok=True)

       model_filename = f"model_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.pt"
       history_filename = f"history_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.png"
       metrics_filename = f"metrics_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.json"
       scalers_filename = f"scalers_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.pkl"

       model_path = os.path.join(fold_dir, model_filename)
       history_path = os.path.join(fold_dir, history_filename)
       metrics_path = os.path.join(fold_dir, metrics_filename)
       scaler_path = os.path.join(fold_dir, scalers_filename)

       # Start training
       print(f"\nStarting training (total {args.epochs} epochs)...")
       # Training history
       train_total_losses = []
       train_mse_losses = []
       train_weighted_losses = []
       train_similarity_losses = []
       train_maes = []
       val_total_losses = []
       val_mse_losses = []
       val_weighted_losses = []
       val_similarity_losses = []
       val_maes = []
       learning_rates = []
       # Early stopping setup
       best_val_loss = float('inf')
       patience_counter = 0
       for epoch in range(args.epochs):
           # Train
           train_total_loss, train_mse_loss, train_weighted_loss, train_similarity_loss, train_mae = train_epoch(
               model, train_loader, criterion, optimizer, device
           )
           # Validate
           val_total_loss, val_mse_loss, val_weighted_loss, val_similarity_loss, val_mae = validate(
               model, val_loader, criterion, device
           )
           # Save current learning rate
           current_lr = optimizer.param_groups[0]['lr']
           learning_rates.append(current_lr)
           # Update learning rate scheduler
           scheduler.step(val_total_loss)
           # Record results
           train_total_losses.append(train_total_loss)
           train_mse_losses.append(train_mse_loss)
           train_weighted_losses.append(train_weighted_loss)
           train_similarity_losses.append(train_similarity_loss)
           train_maes.append(train_mae)
           val_total_losses.append(val_total_loss)
           val_mse_losses.append(val_mse_loss)
           val_weighted_losses.append(val_weighted_loss)
           val_similarity_losses.append(val_similarity_loss)
           val_maes.append(val_mae)
           # Save model (only if validation loss improves)
           if val_total_loss < best_val_loss:
               best_val_loss = val_total_loss
               # Save model with metadata
               torch.save({
                   'epoch': epoch,
                   'model_state_dict': model.state_dict(),
                   'optimizer_state_dict': optimizer.state_dict(),
                   'val_loss': val_total_loss,
                   'train_loss': train_total_loss,
                   'model_params': model_params
               }, model_path)
               patience_counter = 0
               print(f"Epoch {epoch+1}/{args.epochs}, Improved model saved: {os.path.basename(model_path)}")
           else:
               patience_counter += 1
           # Print epoch results
           print(f"Epoch {epoch+1}/{args.epochs}, "
                 f"Train Loss: {train_total_loss:.4f}, Train MAE: {train_mae:.4f}, "
                 f"Val Loss: {val_total_loss:.4f}, Val MAE: {val_mae:.4f}, "
                 f"LR: {current_lr:.6f}")
           # Check for early stopping
           if patience_counter >= args.patience:
               print(f"Early stopping: Validation loss has not improved for {args.patience} consecutive epochs.")
               break
       print("Training completed!")
       # Load best model
       checkpoint = torch.load(model_path)
       model.load_state_dict(checkpoint['model_state_dict'])
       model.eval()
       print(f"\nBest model loaded (Epoch {checkpoint['epoch']+1}, Validation Loss: {checkpoint['val_loss']:.6f})")
       # Evaluate model on test data
       test_total_loss, test_mse_loss, test_weighted_loss, test_similarity_loss, test_mae = validate(
           model, test_loader, criterion, device
       )
       print(f"\nTest total loss: {test_total_loss:.4f}")
       print(f"Test MSE loss: {test_mse_loss:.4f}")
       print(f"Test weighted RGB loss: {test_weighted_loss:.4f}")
       print(f"Test color similarity loss: {test_similarity_loss:.4f}")
       print(f"Test MAE: {test_mae:.4f}")

       # Visualize training results
       plt.figure(figsize=(15, 10))
       # Total loss graph
       plt.subplot(2, 2, 1)
       plt.plot(train_total_losses, label='train_loss')
       plt.plot(val_total_losses, label='val_loss')
       plt.title('Total Loss Function')
       plt.xlabel('Epoch')
       plt.ylabel('Loss')
       plt.legend()
       # MSE loss graph
       plt.subplot(2, 2, 2)
       plt.plot(train_mse_losses, label='train_mse')
       plt.plot(val_mse_losses, label='val_mse')
       plt.title('MSE loss')
       plt.xlabel('Epoch')
       plt.ylabel('Loss')
       plt.legend()
       # Weighted RGB & Color similarity loss graph
       plt.subplot(2, 2, 3)
       plt.plot(train_weighted_losses, label='train_weighted')
       plt.plot(val_weighted_losses, label='val_weighted')
       plt.plot(train_similarity_losses, label='train_similarity')
       plt.plot(val_similarity_losses, label='val_similarity')
       plt.title('"Weighted RGB & Color Similarity Loss"')
       plt.xlabel('Epoch')
       plt.ylabel('Loss')
       plt.legend()
       # MAE & Learning rate graph
       plt.subplot(2, 2, 4)
       ax1 = plt.gca()
       ax1.plot(train_maes, 'b-', label='train_mae')
       ax1.plot(val_maes, 'g-', label='val_mae')
       ax1.set_xlabel('Epoch')
       ax1.set_ylabel('MAE', color='b')
       ax1.tick_params('y', colors='b')
       ax2 = ax1.twinx()
       ax2.plot(learning_rates, 'r-', label='learning_rate')
       ax2.set_ylabel('Learning Rate', color='r')
       ax2.tick_params('y', colors='r')

       lines1, labels1 = ax1.get_legend_handles_labels()
       lines2, labels2 = ax2.get_legend_handles_labels()
       ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
       plt.title('MAE & LR')
       plt.tight_layout()

       plt.savefig(history_path)
       print(f"Training result graph saved: {history_path}")

       print("\nCalculating detailed evaluation metrics and visualizing colors for test data...")
       with torch.no_grad():
           model.eval()
           all_true_rgb = []
           all_pred_rgb = []
           for X_batch, y_batch in test_loader:
               X_batch = X_batch.to(device)
               y_batch = y_batch.cpu().numpy()
               # Run prediction
               outputs = model(X_batch).cpu().numpy()
               # Inverse scaling
               y_batch_original = y_scaler.inverse_transform(y_batch)
               outputs_original = y_scaler.inverse_transform(outputs)
               # Store results
               all_true_rgb.append(y_batch_original)
               all_pred_rgb.append(outputs_original)

           all_true_rgb = np.vstack(all_true_rgb)
           all_pred_rgb = np.vstack(all_pred_rgb)
           # Calculate metrics
           metrics = calculate_metrics(all_true_rgb, all_pred_rgb)
           # Print results
           print("\nEvaluation Metric Results:")
           print(f"R² (R channel): {metrics['r2_r']:.4f}")
           print(f"R² (G channel): {metrics['r2_g']:.4f}")
           print(f"R² (B channel): {metrics['r2_b']:.4f}")
           print(f"RMSE (R channel): {metrics['rmse_r']:.4f}")
           print(f"RMSE (G channel): {metrics['rmse_g']:.4f}")
           print(f"RMSE (B channel): {metrics['rmse_b']:.4f}")
           print(f"RMSE (overall): {metrics['rmse_total']:.4f}")
           if metrics['delta_e_mean'] > 0:
               print(f"Delta E (mean color distance): {metrics['delta_e_mean']:.4f}")
           else:
               print("Delta E: calculation failed")
           # Save metrics
           with open(metrics_path, 'w') as f:
               json.dump(metrics, f, indent=2)
           print(f"Evaluation metrics saved: {metrics_path}")
           # Save scalers
           with open(scaler_path, 'wb') as f:
               pickle.dump({'X_scaler': X_scaler, 'y_scaler': y_scaler}, f)
           print(f"Scalers saved: {scaler_path}")
           # Color visualization - comparing actual colors and predicted colors for test samples
           print("\nVisualizing colors for test samples...")

           colors_vis_filename = f"color_vis_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.png"
           colors_vis_path = os.path.join(fold_dir, colors_vis_filename)
           # Sort by color difference
           color_diffs = np.sqrt(np.sum((all_true_rgb - all_pred_rgb) ** 2, axis=1))
           sorted_indices = np.argsort(color_diffs)[::-1]  # Sort in descending order
           # Number of samples to visualize
           n_samples = min(30, len(all_true_rgb))
           sample_indices = sorted_indices[:n_samples]
           # Visualization setup
           fig, axes = plt.subplots(n_samples, 2, figsize=(8, n_samples * 0.8))
           fig.suptitle(f'Fold {fold+1}: Color Restoration Results (Sorted by Largest Difference)', fontsize=16)

           if n_samples > 1:
               axes[0, 0].set_title('Actual Color\n(Original RGB)')
               axes[0, 1].set_title('Predicted Color\n(Original RGB)')
           else:
               axes[0].set_title('Actual Color\n(Original RGB)')
               axes[1].set_title('Predicted Color\n(Original RGB)')
           # Color comparison visualization
           for i, idx in enumerate(sample_indices):
               # Actual target RGB values
               true_rgb = np.clip(all_true_rgb[idx], 0, 255).astype(np.uint8)
               # Predicted RGB values
               pred_rgb = np.clip(all_pred_rgb[idx], 0, 255).astype(np.uint8)
               # Normalize RGB values to [0, 1] range
               true_rgb_norm = true_rgb / 255.0
               pred_rgb_norm = pred_rgb / 255.0
               # Draw color rectangles
               if n_samples > 1:
                   axes[i, 0].add_patch(plt.Rectangle((0, 0), 1, 1, color=true_rgb_norm))
                   axes[i, 1].add_patch(plt.Rectangle((0, 0), 1, 1, color=pred_rgb_norm))
                   for j in range(2):
                       axes[i, j].set_xticks([])
                       axes[i, j].set_yticks([])
                       axes[i, j].set_xlim(0, 1)
                       axes[i, j].set_ylim(0, 1)
                   # Add RGB value text
                   axes[i, 0].text(0.5, -0.1, f"({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})",
                                  ha='center', va='center', fontsize=8, transform=axes[i, 0].transAxes)
                   axes[i, 1].text(0.5, -0.1, f"({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})",
                                  ha='center', va='center', fontsize=8, transform=axes[i, 1].transAxes)
               else:
                   # For single sample
                   axes[0].add_patch(plt.Rectangle((0, 0), 1, 1, color=true_rgb_norm))
                   axes[1].add_patch(plt.Rectangle((0, 0), 1, 1, color=pred_rgb_norm))
                   for j in range(2):
                       axes[j].set_xticks([])
                       axes[j].set_yticks([])
                       axes[j].set_xlim(0, 1)
                       axes[j].set_ylim(0, 1)
                   # Add RGB value text
                   axes[0].text(0.5, -0.1, f"({true_rgb[0]}, {true_rgb[1]}, {true_rgb[2]})",
                               ha='center', va='center', fontsize=8, transform=axes[0].transAxes)
                   axes[1].text(0.5, -0.1, f"({pred_rgb[0]}, {pred_rgb[1]}, {pred_rgb[2]})",
                               ha='center', va='center', fontsize=8, transform=axes[1].transAxes)
           plt.tight_layout()
           plt.subplots_adjust(top=0.95)
           plt.savefig(colors_vis_path, dpi=300, bbox_inches='tight')
           plt.close(fig)
           print(f"Color visualization saved: {colors_vis_path}")
           # Save color sample data to CSV
           samples_data = []
           for idx in sample_indices:
               true_rgb = all_true_rgb[idx]
               pred_rgb = all_pred_rgb[idx]
               color_diff = color_diffs[idx]
               samples_data.append({
                   'sample_idx': idx,
                   'true_r': int(np.clip(true_rgb[0], 0, 255)),
                   'true_g': int(np.clip(true_rgb[1], 0, 255)),
                   'true_b': int(np.clip(true_rgb[2], 0, 255)),
                   'pred_r': int(np.clip(pred_rgb[0], 0, 255)),
                   'pred_g': int(np.clip(pred_rgb[1], 0, 255)),
                   'pred_b': int(np.clip(pred_rgb[2], 0, 255)),
                   'color_diff': float(color_diff)
               })
           samples_df = pd.DataFrame(samples_data)
           samples_csv_filename = f"color_samples_fold{fold+1}_e{args.epochs}_b{args.batch_size}_lr{args.learning_rate}_lat{args.latent_dim}.csv"
           samples_csv_path = os.path.join(fold_dir, samples_csv_filename)
           samples_df.to_csv(samples_csv_path, index=False)
           print(f"Sample color data saved as CSV: {samples_csv_path}")
           # Save fold results
           fold_result = {
               'fold': fold + 1,
               'test_total_loss': float(test_total_loss),
               'test_mse_loss': float(test_mse_loss),
               'test_weighted_loss': float(test_weighted_loss),
               'test_similarity_loss': float(test_similarity_loss),
               'test_mae': float(test_mae),
               'r2_mean': float((metrics['r2_r'] + metrics['r2_g'] + metrics['r2_b']) / 3),
               'rmse_mean': float((metrics['rmse_r'] + metrics['rmse_g'] + metrics['rmse_b']) / 3),
               'delta_e_mean': float(metrics['delta_e_mean']),
               'best_epoch': int(checkpoint['epoch'] + 1)
           }
           fold_results.append(fold_result)
           fold_test_losses.append(test_total_loss)
           fold_test_maes.append(test_mae)
           fold_r2_scores.append((metrics['r2_r'] + metrics['r2_g'] + metrics['r2_b']) / 3)
           fold_rmse_scores.append((metrics['rmse_r'] + metrics['rmse_g'] + metrics['rmse_b']) / 3)
           fold_delta_e_means.append(metrics['delta_e_mean'])

   # 8. K-Fold results summary
   print("\n" + "="*60)
   print("K-Fold Cross Validation Results Summary")
   print("="*60)
   # Calculate average metrics
   avg_test_loss = np.mean(fold_test_losses)
   avg_test_mae = np.mean(fold_test_maes)
   avg_r2 = np.mean(fold_r2_scores)
   avg_rmse = np.mean(fold_rmse_scores)
   avg_delta_e = np.mean(fold_delta_e_means)
   # Calculate standard deviations
   std_test_loss = np.std(fold_test_losses)
   std_test_mae = np.std(fold_test_maes)
   std_r2 = np.std(fold_r2_scores)
   std_rmse = np.std(fold_rmse_scores)
   std_delta_e = np.std(fold_delta_e_means)
   # Print results for each fold
   print("\nResults by fold:")
   print("-"*80)
   print(f"{'Fold':^6} | {'Test Loss':^12} | {'Test MAE':^10} | {'R² Avg':^8} | {'RMSE Avg':^10} | {'Delta E':^8} | {'Best Epoch':^10}")
   print("-"*80)
   for result in fold_results:
       print(f"{result['fold']:^6} | {result['test_total_loss']:^12.4f} | {result['test_mae']:^10.4f} | "
             f"{result['r2_mean']:^8.4f} | {result['rmse_mean']:^10.4f} | "
             f"{result['delta_e_mean']:^8.4f} | {result['best_epoch']:^10}")
   print("-"*80)
   # Print average metrics
   print("\nOverall metrics across all folds:")
   print(f"Test Loss: {avg_test_loss:.4f} ± {std_test_loss:.4f}")
   print(f"Test MAE: {avg_test_mae:.4f} ± {std_test_mae:.4f}")
   print(f"R² Average: {avg_r2:.4f} ± {std_r2:.4f}")
   print(f"RMSE Average: {avg_rmse:.4f} ± {std_rmse:.4f}")
   print(f"Delta E Average: {avg_delta_e:.4f} ± {std_delta_e:.4f}")
   # Save summary metrics
   summary = {
       'avg_test_loss': float(avg_test_loss),
       'avg_test_mae': float(avg_test_mae),
       'avg_r2': float(avg_r2),
       'avg_rmse': float(avg_rmse),
       'avg_delta_e': float(avg_delta_e),
       'std_test_loss': float(std_test_loss),
       'std_test_mae': float(std_test_mae),
       'std_r2': float(std_r2),
       'std_rmse': float(std_rmse),
       'std_delta_e': float(std_delta_e),
       'fold_results': fold_results
   }
   summary_path = os.path.join(output_dir, "kfold_summary.json")
   with open(summary_path, 'w') as f:
       json.dump(summary, f, indent=2)
   print(f"\nK-Fold summary information saved: {summary_path}")
   # Fold comparison visualization
   plt.figure(figsize=(15, 10))
   # Metrics to visualize
   metrics_names = ['Test Loss', 'Test MAE', 'R² Average', 'RMSE Average', 'Delta E']
   metrics_values = [fold_test_losses, fold_test_maes, fold_r2_scores, fold_rmse_scores, fold_delta_e_means]
   metrics_avgs = [avg_test_loss, avg_test_mae, avg_r2, avg_rmse, avg_delta_e]
   for i, (name, values, avg) in enumerate(zip(metrics_names, metrics_values, metrics_avgs)):
       plt.subplot(2, 3, i+1)
       # Bar graph of individual fold values
       plt.bar(range(1, args.n_folds+1), values, alpha=0.7)
       # Average line
       plt.axhline(y=avg, color='r', linestyle='-', label=f'Average: {avg:.4f}')
       plt.title(name)
       plt.xlabel('Fold')
       plt.xticks(range(1, args.n_folds+1))
       plt.legend()
   plt.tight_layout()
   # Save comparison graph
   comparison_path = os.path.join(output_dir, "kfold_comparison.png")
   plt.savefig(comparison_path)
   print(f"Fold comparison visualization saved: {comparison_path}")
   print("\n" + "="*60)
   print("K-Fold cross-validation completed successfully")
   print("="*60)
   return summary

##Train

In [ ]:
```python
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Color Calibration AutoEncoder Model K-Fold Cross Validation')
    # Data-related arguments
    parser.add_argument('--master_path', type=str,
                     default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/full_master_data.csv',
                     help='Master dataset path')
    parser.add_argument('--output_dir', type=str,
                     default='/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/output/kfold_model',
                     help='Model save directory')
    parser.add_argument('--use_drive', action='store_true', help='Use Google Drive')
    parser.add_argument('--test_size', type=float, default=0.2, help='Test set ratio')
    # K-Fold settings
    parser.add_argument('--n_folds', type=int, default=5, help='Number of cross-validation folds')
    # Model configuration arguments
    parser.add_argument('--latent_dim', type=int, default=32, help='Latent space dimension')
    parser.add_argument('--encoder_layers', type=int, default=3, choices=[2, 3], help='Number of encoder hidden layers')
    parser.add_argument('--decoder_layers', type=int, default=3, choices=[2, 3], help='Number of decoder hidden layers')
    parser.add_argument('--dropout_rate', type=float, default=0.2, help='Dropout rate')
    # Training arguments
    parser.add_argument('--epochs', type=int, default=100, help='Number of training epochs')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
    parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--patience', type=int, default=10, help='Early stopping patience')
    parser.add_argument('--seed', type=int, default=42, help='Random seed')
    parser.add_argument('--no_cuda', action='store_true', help='Do not use CUDA')
    # Loss function weights
    parser.add_argument('--loss_alpha', type=float, default=0.2, help='MSE loss weight')
    parser.add_argument('--loss_beta', type=float, default=0.5, help='Weighted RGB loss weight')
    parser.add_argument('--loss_gamma', type=float, default=0.3, help='Color similarity loss weight')
    # Modifications for running in Colab
    import sys
    if 'google.colab' in sys.modules:
        args = parser.parse_args([
            '--master_path', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/full_master_data.csv',
            '--output_dir', '/content/drive/MyDrive/2025_COSI_149B_Project1/Teamwork/output/kfold_model',
            '--use_drive',
            '--n_folds', '5',
            '--epochs', '100',
            '--batch_size', '32',
            '--learning_rate', '0.001',
            '--latent_dim', '512',
            '--encoder_layers', '2',
            '--decoder_layers', '3',
            '--dropout_rate', '0.2'
        ])
    else:
        args = parser.parse_args()
    # Verify that alpha + beta + gamma = 1
    sum_weights = args.loss_alpha + args.loss_beta + args.loss_gamma
    if abs(sum_weights - 1.0) > 1e-5:
        print(f"Warning: Sum of loss function weights is not 1 (current: {sum_weights})")
        # Normalize weights
        args.loss_alpha /= sum_weights
        args.loss_beta /= sum_weights
        args.loss_gamma /= sum_weights
        print(f"Weights have been normalized: alpha={args.loss_alpha:.4f}, beta={args.loss_beta:.4f}, gamma={args.loss_gamma:.4f}")
    # Record start time
    from datetime import datetime
    start_time = datetime.now()
    print(f"K-Fold cross-validation start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    # Run K-Fold cross-validation
    summary = main_from_master_dataset(args)
    # Print end time and elapsed time
    end_time = datetime.now()
    elapsed_time = end_time - start_time
    print(f"K-Fold cross-validation end time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total elapsed time: {elapsed_time}")
    # Print results summary
    print("\n===== K-Fold Cross-Validation Results Summary =====")
    print(f"Average test loss: {summary['avg_test_loss']:.4f} ± {summary['std_test_loss']:.4f}")
    print(f"Average test MAE: {summary['avg_test_mae']:.4f} ± {summary['std_test_mae']:.4f}")
    print(f"Average R²: {summary['avg_r2']:.4f} ± {summary['std_r2']:.4f}")
    print(f"Average RMSE: {summary['avg_rmse']:.4f} ± {summary['std_rmse']:.4f}")
    print(f"Average Delta E: {summary['avg_delta_e']:.4f} ± {summary['std_delta_e']:.4f}")
    print("===================================================")
    # Find the best fold
    best_fold = min(summary['fold_results'], key=lambda x: x['test_total_loss'])
    print(f"\nBest model fold: {best_fold['fold']}")
    print(f"Best model test loss: {best_fold['test_total_loss']:.4f}")
    print(f"Best model R² average: {best_fold['r2_mean']:.4f}")
    print(f"Best model path: {args.output_dir}/master_kfold_{args.n_folds}_*/fold_{best_fold['fold']}/")
```